In [ ]:
# ▶ Colab setup — run this cell first. (On BinderHub it does nothing.)
import sys, os
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/gen-hep-notebooks'):
        !git clone -b deploy --depth 1 https://github.com/livaage/gen-hep-notebooks.git /content/gen-hep-notebooks
        !pip install -q -r /content/gen-hep-notebooks/requirements-colab.txt
    sys.path.insert(0, '/content/gen-hep-notebooks')
    os.chdir('/content/gen-hep-notebooks/notebooks')

<div style="border-left: 4px solid #2563eb; background: #f8fafc; padding: 14px 18px; margin: 6px 0; border-radius: 4px; color: #1f2937;"><div style="font-weight: 600; color: #1e3a8a; margin-bottom: 8px; font-size: 1.05em;">👋 Welcome — quick reference for Cadence</div><ul style="margin: 0; padding-left: 22px; line-height: 1.65; color: #1f2937;"><li><strong>Submit an answer:</strong> each exercise cell ends with <code>check("id", answer)</code> — replace the placeholder with your answer and run the cell.</li><li><strong>Submit a plot:</strong> <code>submit_image("id", fig)</code> ships a matplotlib or Plotly figure to your teacher's dashboard.</li><li><strong>Free-text / reflections:</strong> write your response, then <code>mark_done("id")</code> to mark it complete.</li><li><strong>Stuck?</strong> <code>show_hint("id")</code> for the teacher's hint, or <code>show_solution("id")</code> after a few wrong attempts if your teacher enabled solution reveals.</li><li><strong>Your data:</strong> <code>%cadence_export_my_data</code> to download what's stored about you, <code>%cadence_delete_my_data --yes</code> to wipe it.</li></ul></div>

In [ ]:
%load_ext cadence
%cadence_session ginger-glacier-87 "your name"
from cadence import check, show_hint, show_solution, mark_done, submit_image

# 05 · Evaluating Generative Models

**Generative Modelling for HEP** — notebook 5 of 6.

You've built a VAE (01), a GAN (02) and a diffusion model (03). Each *produces*
jets — but **how do you know any of them is good?** "The samples look nice" is
not a metric. In HEP we need numbers that compare a generated distribution to
the real one on the observables physicists care about.

This notebook is the **scoring toolkit**:

1. **W1** — the 1-Wasserstein distance on the jet-mass spectrum. Cheap,
   interpretable, per-observable.
2. **FPD** and **KPD** — Frechet / Kernel Physics Distances from
   `jetnet.evaluation`: single-number, distribution-level scores that have
   become the JetNet community standard.
3. A **classifier two-sample test** — we train a real **GNN** to discriminate
   real from generated jets. If the best classifier we can train still only
   reaches **AUC ≈ 0.5**, the two samples are statistically indistinguishable —
   the strongest evidence a generator is good.

> You complete the cells marked **Exercise** — most ask for a number, a couple
> ask for a short **written reflection**. Everything else runs as-is.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys; sys.path.insert(0, "..")
from src.seeds import set_seed
from src.data import load_jetnet
from src.jetmass import jet_mass, plot_jet_mass, jet_mass_w1

SEED = 0
N_JETS = 5000
N_PARTICLES = 30
set_seed(SEED)
rng = np.random.default_rng(SEED)
print(f"seed={SEED}  n_jets={N_JETS}  n_particles={N_PARTICLES}")

## Part A · A bench of synthetic "generators"

We load real gluon jets, then fabricate two stand-in generators so the metrics
have something to rank — no trained model needed:

* **`good`** — real jets with a *tiny* per-particle jitter. A near-perfect
  generator: distributions should overlap almost exactly.
* **`bad`** — real jets with *heavy* jitter **and** a clipped high-mass tail
  (the classic failure mode: generators struggle with the rare, high-mass
  jets). Distributions should visibly disagree.

A good evaluation metric must **rank `good` above `bad`**.

In [ ]:
real = load_jetnet("g", num_particles=N_PARTICLES, max_jets=N_JETS).astype(np.float32)


def perturb(jets, scale, seed):
    """Add Gaussian jitter to real particles (pt_rel>0), keeping padding at 0."""
    r = np.random.default_rng(seed)
    out = jets.copy()
    mask = out[..., 2] > 0                       # real (non-padded) particles
    noise = r.normal(0.0, scale, size=out.shape).astype(np.float32)
    out += noise * mask[..., None]
    out[..., 2] = np.clip(out[..., 2], 0.0, None)  # pt_rel stays non-negative
    return out


def clip_high_mass(jets, keep_quantile=0.6):
    """Drop the high-mass tail and resample from the bulk — mimics a generator
    that never learns the rare high-mass jets."""
    m = jet_mass(jets)
    cut = np.quantile(m, keep_quantile)
    bulk = jets[m <= cut]
    idx = np.random.default_rng(SEED).integers(0, len(bulk), size=len(jets))
    return bulk[idx]


# split real into a reference half and an "as-if-real" eval half
half = len(real) // 2
real_ref, real_eval = real[:half], real[half:]

good = perturb(real_eval, scale=0.003, seed=1)               # near-perfect
bad = clip_high_mass(perturb(real_eval, scale=0.03, seed=2)) # heavy + clipped tail

generators = {"good": good, "bad": bad}
print("real_ref:", real_ref.shape, "| good:", good.shape, "| bad:", bad.shape)

In [ ]:
# Eyeball the spine first: real vs good vs bad jet mass.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_jet_mass(real_ref, good, labels=("real", "good gen"),
              title="Good generator", ax=axes[0])
plot_jet_mass(real_ref, bad, labels=("real", "bad gen"),
              title="Bad generator", ax=axes[1])
plt.tight_layout(); plt.show()
print("plotted jet-mass overlays for both generators")

<!-- cadence:task ex1-w1 -->
### Exercise 1 — W1 on the jet mass

The simplest score: the **1-Wasserstein distance** between the real and
generated jet-mass distributions (`src.jetmass.jet_mass_w1`). It's the average
"earth-mover" cost to morph one histogram into the other — **smaller is
better**, and it's in the same units as the observable.

Compute the W1 between `real_ref` and the **good** generator and return it as
`w1_good` (a number).

In [ ]:
# the 1-Wasserstein distance between real and the GOOD generator's jet mass:
w1_good = ...
print("W1(real, good) =", w1_good)
print("W1(real, bad)  =", round(jet_mass_w1(real_ref, bad), 5))   # the bad one should score worse

check("ex1-w1", ...)

## Part B · FPD and KPD

W1 scores **one observable at a time**. The JetNet community standard is to
collapse the whole jet into a feature vector and compare the *distributions* of
those features:

* **FPD** (Frechet Physics Distance) — like FID for images: fits Gaussians to
  the real and generated feature clouds and measures the Frechet distance.
  Lower = better; an unbiased extrapolation removes finite-sample bias.
* **KPD** (Kernel Physics Distance) — a kernel (MMD-style) two-sample distance,
  more robust to outliers. Lower = better.

`jetnet.evaluation` provides `fpd` and `kpd` directly. We compute a small set of
physics-motivated features per jet (the **EFPs**-style summaries: mass plus a
few momentum moments) and feed those in. If `jetnet` isn't installed we fall
back to a simple Frechet distance on the same features, so the cell still runs.

In [ ]:
def jet_features(jets):
    """A compact per-jet feature vector for distribution-level metrics:
    jet mass + summary statistics of the particle kinematics. Shape (N, F)."""
    m = jet_mass(jets)[:, None]
    pt = jets[..., 2]
    eta, phi = jets[..., 0], jets[..., 1]
    n_active = (pt > 0).sum(axis=1, keepdims=True).astype(np.float32)
    pt_sum = pt.sum(axis=1, keepdims=True)
    pt_max = pt.max(axis=1, keepdims=True)
    eta_w = (np.abs(eta) * (pt > 0)).sum(axis=1, keepdims=True)
    phi_w = (np.abs(phi) * (pt > 0)).sum(axis=1, keepdims=True)
    return np.concatenate([m, n_active, pt_sum, pt_max, eta_w, phi_w], axis=1).astype(np.float64)


def frechet_distance(a, b):
    """Plain Frechet distance between two Gaussians fit to feature clouds —
    the no-jetnet fallback for FPD."""
    mu_a, mu_b = a.mean(0), b.mean(0)
    ca, cb = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    diff = mu_a - mu_b
    # sqrt of product of covariances via eigen-decomposition (symmetric PSD)
    prod = ca @ cb
    eigvals = np.linalg.eigvals(prod)
    covmean = np.sqrt(np.clip(eigvals.real, 0, None)).sum()
    return float(diff @ diff + np.trace(ca) + np.trace(cb) - 2 * covmean)


def fpd_score(a, b):
    """FPD between two feature clouds — prefer jetnet.evaluation.fpd (bias-corrected;
    it returns (value, error), we keep the value), else the frechet_distance
    fallback. Using ONE function for every generator keeps the scores comparable."""
    try:
        from jetnet.evaluation import fpd
        return float(fpd(a, b)[0])
    except Exception:
        return frechet_distance(a, b)


feats_real = jet_features(real_ref)
feats_good = jet_features(good)
feats_bad = jet_features(bad)
print("feature vectors:", feats_real.shape, "(jet mass + 5 kinematic summaries)")

<!-- cadence:task ex2-fpd -->
### Exercise 2 — FPD for the bad generator

W1 scores one observable at a time; **FPD** (Frechet Physics Distance) collapses
the whole jet into a feature vector and compares the *distributions* — lower is
better. Compute the FPD between the real features and the **bad** generator's
features with the `fpd_score` helper and return it as `fpd_bad`.

(We expect `fpd_bad` to be clearly *larger* than the good generator's FPD — the
same helper scores both, so the ordering is a fair comparison.)

In [ ]:
# the FPD between the real features and the BAD generator's (use the fpd_score helper):
fpd_bad = ...
print("FPD(real, bad)  =", fpd_bad)
print("FPD(real, good) =", round(fpd_score(feats_real, feats_good), 5))   # good should be lower

check("ex2-fpd", ...)

<!-- cadence:task ex3-rank -->
### Exercise 3 — Rank the generators

You now have two scores per generator. A metric is only useful if it **orders
the generators correctly**: the good one should beat the bad one. Compute FPD
(or W1, your choice — they should agree here) for both generators, and return
the **name of the better generator** as the string `best_generator` — either
`"good"` or `"bad"`.

In [ ]:
# score each generator with the SAME metric (lower = better), then pick the winner:
scores = {
    "good": ...,   # fpd_score(feats_real, feats_good)
    "bad": ...,    # fpd_score(feats_real, feats_bad)
}
best_generator = ...   # the key with the SMALLEST score: "good" or "bad"
print("scores:", {k: round(v, 5) for k, v in scores.items()}, "| best:", best_generator)

check("ex3-rank", ...)

## Part C · The classifier two-sample test

The gold standard. If a generator is perfect, then **no classifier**, however
powerful, can tell its samples from real ones — the best achievable ROC **AUC is
0.5** (pure chance). The further the AUC climbs toward 1.0, the more a
discriminator has found a tell-tale difference, i.e. the worse the generator.

We train a small **permutation-invariant GNN** — a **DeepSets** encoder over the
jet point cloud (particle order doesn't matter) — to classify *real (label 0)*
vs *generated (label 1)*, then read its **AUC** on a held-out split. It trains in
a few seconds on CPU.

In [ ]:
import torch
import torch.nn as nn
from src.train import get_device, train
from src.gnn import DeepSetsEncoder

device = get_device()
GNN_EPOCHS = 5
GNN_BATCH = 128


class JetClassifier(nn.Module):
    """Real-vs-generated discriminator: a permutation-invariant DeepSets encoder
    over the jet point cloud -> a single logit."""

    def __init__(self, latent=32):
        super().__init__()
        self.encoder = DeepSetsEncoder(in_features=3, hidden=64, latent=latent)
        self.head = nn.Linear(latent, 1)

    def forward(self, x):
        return self.head(self.encoder(x)).squeeze(-1)


def make_dataset(real_jets, gen_jets):
    """Stack real (0) and generated (1) jets into one labelled tensor set."""
    x = np.concatenate([real_jets, gen_jets]).astype(np.float32)
    y = np.concatenate([np.zeros(len(real_jets)), np.ones(len(gen_jets))]).astype(np.float32)
    perm = np.random.default_rng(SEED).permutation(len(x))
    return x[perm], y[perm]


def roc_auc(y_true, scores):
    """Rank-based ROC AUC (Mann-Whitney U) — no sklearn dependency."""
    y_true = np.asarray(y_true); scores = np.asarray(scores)
    order = np.argsort(scores)
    ranks = np.empty_like(order, dtype=np.float64)
    ranks[order] = np.arange(1, len(scores) + 1)
    n_pos = y_true.sum(); n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    auc = (ranks[y_true == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    return float(auc)


print("JetClassifier ready (DeepSets encoder) "
      f"| epochs={GNN_EPOCHS} batch={GNN_BATCH} device={device}")

<!-- cadence:task ex4-classifier -->
### Exercise 4 — Train the classifier, read the AUC

Build the real-vs-generated dataset for the **good** generator, train the
`JetClassifier` with a binary-cross-entropy objective, and read the **ROC AUC**
on a held-out split. Return it as `auc_good`.

A near-perfect generator should drive the AUC down toward **0.5**. (We provide
`make_dataset`, `roc_auc`, and a `train(...)` loop; you fill the BCE step and the
scoring.)

In [ ]:
X, Y = make_dataset(real_ref, good)
n_train = int(0.8 * len(X))
Xtr, Ytr = X[:n_train], Y[:n_train]
Xte, Yte = X[n_train:], Y[n_train:]

clf = JetClassifier().to(device)
tr_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.as_tensor(Xtr), torch.as_tensor(Ytr)),
    batch_size=GNN_BATCH, shuffle=True)


def bce_step(model, batch):
    x, y = batch
    logits = model(x)
    # binary cross-entropy between the predicted logits and the labels y:
    return ...

train(clf, tr_loader, bce_step, epochs=GNN_EPOCHS, lr=1e-3, device=device)

clf.eval()
with torch.no_grad():
    # score the held-out jets Xte (a probability per jet), then read the ROC AUC:
    scores = ...
auc_good = round(roc_auc(Yte, scores), 3)
print("classifier two-sample-test AUC (good generator):", auc_good)

check("ex4-classifier", ...)

<!-- cadence:task ex5-verdict -->
### Exercise 5 — Read the verdict (reflection)

The counterintuitive one. A classifier two-sample test on a **near-perfect**
generator gives an AUC close to **0.5**: the discriminator does no better than a
coin flip, so the generated jets are statistically indistinguishable from real
ones. Write 2–3 sentences and submit:

- Why does an AUC near **0.5** mean a *good* generator, while AUC → 1.0 means a
  *bad* one? (It's the opposite of most ML metrics — say why.)
- Your `auc_good` probably didn't land exactly at 0.5. What does a small excess
  above 0.5 tell you the classifier managed to find?

In [ ]:
# Your reflection (2-3 sentences), then run the cell to submit:
#   * why does AUC ~ 0.5 mean a GOOD (indistinguishable) generator, and AUC -> 1 a
#     bad one? (the opposite of most ML metrics)
#   * what does a small excess above 0.5 tell you the classifier found?

check("ex5-verdict", ...)

<!-- cadence:task ex6-disagree -->
### Exercise 6 — When do the metrics disagree? (reflection)

W1, FPD, and the classifier test all ranked `good` above `bad` here — but they
measure different things. Write 2–3 sentences and submit:

- W1 scores **one observable** (jet mass) at a time; FPD and the classifier test
  see the **whole** feature vector. Describe a generator that could look great on
  the jet-mass W1 but get caught by FPD or the classifier.
- If your three metrics *disagreed* on the ranking, what would that tell you about
  the generator?

In [ ]:
# Your reflection (2-3 sentences), then run the cell to submit:
#   * describe a generator that scores well on jet-mass W1 but is caught by FPD or
#     the classifier test (which see the whole feature vector).
#   * if the three metrics disagreed on the ranking, what would that tell you?

check("ex6-disagree", ...)

## Recap

- **No single number is enough** — score generators on the observables that
  matter, with complementary tools.
- **W1** is cheap and per-observable; great for the **jet-mass spine** we've
  tracked since notebook 01.
- **FPD / KPD** (`jetnet.evaluation`) give distribution-level, single-number
  scores — the JetNet community standard. **Lower is better**; a high FPD means
  the generated feature distribution is far from real.
- The **classifier two-sample test** is the strongest test: train the best
  discriminator you can and read its **AUC**. **AUC ≈ 0.5 = indistinguishable =
  good**; AUC → 1.0 means the generator has a tell.
- All three should **rank generators consistently** — if they disagree, you've
  learned something about *which* features your generator gets wrong.